<a href="https://colab.research.google.com/github/vibhorjoshi/-CHECK/blob/main/amazon%20ml%20challenge%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install sentence-transformers timm transformers
!pip install pandas numpy scikit-learn lightgbm xgboost umap-learn gensim nltk tqdm

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4 --force-reinstall
!pip install --upgrade pandas scikit-learn torch


In [ ]:
import tensorflow as tf
print("Num GPUs:", len(tf.config.list_physical_devices('GPU')))
print("GPU Device:", tf.test.gpu_device_name())

In [ ]:
# Set CUDA environment
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Check GPU availability
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print(f"GPU detected: {physical_devices}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("No GPU detected, using CPU. Please ensure GPU runtime is enabled in Colab (Runtime > Change runtime type > GPU).")
    print("Run the following to verify GPU availability:")
    print("!nvidia-smi")
    print("If issues persist, try disconnecting and reconnecting the runtime or upgrading to Colab Pro.")


In [ ]:

pip install numpy==1.26.4 --force-reinstall


In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from utils import download_images  # Use provided utils.py

# Check GPU availability
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print(f"GPU detected: {physical_devices}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("No GPU detected, using CPU")

# Step 1: Data Cleaning & Validation
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Regex patterns
title_pattern = r'Item Name: ([^\n]+)'
value_pattern = r'Value: ([0-9.]+)'
unit_pattern = r'Unit: ([^\n]+)'

def parse_catalog_content(content):
    title = re.search(title_pattern, content)
    value = re.search(value_pattern, content)
    unit = re.search(unit_pattern, content)

    title = title.group(1) if title else ''
    value = float(value.group(1)) if value else 1.0
    unit = unit.group(1) if unit else 'Unknown'
    description = re.sub(title_pattern, '', content)
    description = re.sub(value_pattern, '', description)
    description = re.sub(unit_pattern, '', description).strip()

    return {'title': title, 'description': description, 'ipq': value, 'unit': unit}

def safe_parse(row):
    try:
        result = parse_catalog_content(row['catalog_content'])
        if not isinstance(result, dict):
            print(f"Warning: parse_catalog_content returned {type(result)} for sample_id {row['sample_id']}")
            return {'title': '', 'description': '', 'ipq': 1.0, 'unit': 'Unknown'}
        return result
    except Exception as e:
        print(f"Error parsing sample_id {row['sample_id']}: {e}")
        return {'title': '', 'description': '', 'ipq': 1.0, 'unit': 'Unknown'}

train_df['parsed'] = train_df.apply(safe_parse, axis=1)
test_df['parsed'] = test_df.apply(safe_parse, axis=1)

def extract_field(df, field):
    return df['parsed'].apply(lambda x: x[field] if isinstance(x, dict) else (1.0 if field == 'ipq' else ''))

train_df['title'] = extract_field(train_df, 'title')
train_df['description'] = extract_field(train_df, 'description')
train_df['ipq'] = extract_field(train_df, 'ipq')
train_df['unit'] = extract_field(train_df, 'unit')
test_df['title'] = extract_field(test_df, 'title')
test_df['description'] = extract_field(test_df, 'description')
test_df['ipq'] = extract_field(test_df, 'ipq')
test_df['unit'] = extract_field(test_df, 'unit')

train_df = train_df.drop('parsed', axis=1)
test_df = test_df.drop('parsed', axis=1)

# Handle missing values before creating 'text' column
train_df['title'] = train_df['title'].fillna('missing')
train_df['description'] = train_df['description'].fillna('missing')
test_df['title'] = test_df['title'].fillna('missing')
test_df['description'] = test_df['description'].fillna('missing')

# Create 'text' column
train_df['text'] = train_df['title'] + ' ' + train_df['description']
test_df['text'] = test_df['title'] + ' ' + test_df['description']

# Validate 'text' column
if 'text' not in test_df.columns:
    raise KeyError("Failed to create 'text' column in test_df")
train_df['text'] = train_df['text'].fillna('missing')
test_df['text'] = test_df['text'].fillna('missing')

# Validate prices
train_df = train_df[train_df['price'] > 0]
Q1 = train_df['price'].quantile(0.25)
Q3 = train_df['price'].quantile(0.75)
IQR = Q3 - Q1
train_df = train_df[(train_df['price'] >= Q1 - 1.5 * IQR) & (train_df['price'] <= Q3 + 1.5 * IQR)]

print("✅ Text and IPQ fields prepared")
print("✅ Text and IPQ fields prepared")

In [ ]:
# Step 2: Unsupervised Ecosystem Discovery
text_model = SentenceTransformer('all-MiniLM-L6-v2')
train_text_embeddings = text_model.encode(train_df['text'].tolist(), batch_size=32, show_progress_bar=True, device='cuda' if len(physical_devices) > 0 else 'cpu')
test_text_embeddings = text_model.encode(test_df['text'].tolist(), batch_size=32, show_progress_bar=True, device='cuda' if len(physical_devices) > 0 else 'cpu')

kmeans = KMeans(n_clusters=30, random_state=42)
train_df['cluster'] = kmeans.fit_predict(train_text_embeddings)
test_df['cluster'] = kmeans.predict(test_text_embeddings)

pca = PCA(n_components=50, random_state=42)
train_pca = pca.fit_transform(train_text_embeddings)
test_pca = pca.transform(test_text_embeddings)
train_df[[f'pca_{i}' for i in range(50)]] = train_pca
test_df[[f'pca_{i}' for i in range(50)]] = test_pca

vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
text_matrix = vectorizer.fit_transform(train_df['text'])
lda = LatentDirichletAllocation(n_components=20, random_state=42)
train_topics = lda.fit_transform(text_matrix)
test_topics = lda.transform(vectorizer.transform(test_df['text']))
train_df[[f'topic_{i}' for i in range(20)]] = train_topics
test_df[[f'topic_{i}' for i in range(20)]] = test_topics


def extract_brand(title):
    words = title.split()
    return words[0] if words else 'Unknown'
train_df['brand'] = train_df['title'].apply(extract_brand)
test_df['brand'] = test_df['title'].apply(extract_brand)
brand_freq = train_df['brand'].value_counts().to_dict()
train_df['brand_freq'] = train_df['brand'].map(brand_freq)
test_df['brand_freq'] = test_df['brand'].map(brand_freq).fillna(brand_freq.mean())



In [ ]:
from pathlib import Path

# Step 3: Multi-Modal Feature Extraction
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
train_df['char_count'] = train_df['text'].apply(len)
train_df['title_desc_ratio'] = train_df['title'].apply(len) / (train_df['description'].apply(len) + 1)
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))
test_df['char_count'] = test_df['text'].apply(len)
test_df['title_desc_ratio'] = test_df['title'].apply(len) / (test_df['description'].apply(len) + 1)

image_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
def extract_image_features(image_path):
    try:
        img = load_img(image_path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        img_array = np.expand_dims(img_array, axis=0)
        with tf.device('/GPU:0' if len(physical_devices) > 0 else '/CPU:0'):
            features = image_model.predict(img_array, verbose=0)
        return features.flatten()
    except:
        return np.zeros(512)

# Use download_images from utils.py
download_images(train_df['image_link'].tolist(), 'train_images')
download_images(test_df['image_link'].tolist(), 'test_images')
train_image_features = np.array([extract_image_features(f'/content/train_images/{Path(url).name}') for url in train_df['image_link']])
test_image_features = np.array([extract_image_features(f'/content/test_images{Path(url).name}') for url in test_df['image_link']])

scaler_text = StandardScaler()
scaler_image = StandardScaler()
scaler_tabular = StandardScaler()
train_text_embeddings = scaler_text.fit_transform(train_text_embeddings)
test_text_embeddings = scaler_text.transform(test_text_embeddings)
train_image_features = scaler_image.fit_transform(train_image_features)
test_image_features = scaler_image.transform(test_image_features)

tabular_features = ['ipq', 'brand_freq', 'word_count', 'char_count', 'title_desc_ratio', 'cluster'] + \
                   [f'pca_{i}' for i in range(50)] + [f'topic_{i}' for i in range(20)]
train_df['ipq_log'] = np.log1p(train_df['ipq'])
test_df['ipq_log'] = np.log1p(test_df['ipq'])
tabular_features.append('ipq_log')
train_tabular = scaler_tabular.fit_transform(train_df[tabular_features])
test_tabular = scaler_tabular.transform(test_df[tabular_features])



In [ ]:
# Step 4: M2TFM
class M2TFM(Model):
    def __init__(self, text_dim=384, image_dim=512, tabular_dim=len(tabular_features), hidden_dim=256, num_heads=8, num_blocks=3):
        super(M2TFM, self).__init__()
        self.text_projection = layers.Dense(hidden_dim, activation='relu')
        self.image_projection = layers.Dense(hidden_dim, activation='relu')
        self.tabular_projection = layers.Dense(hidden_dim, activation='relu')
        self.text_embed = tf.Variable(tf.random.normal([1, hidden_dim]), trainable=True)
        self.image_embed = tf.Variable(tf.random.normal([1, hidden_dim]), trainable=True)
        self.tabular_embed = tf.Variable(tf.random.normal([1, hidden_dim]), trainable=True)
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=hidden_dim)
        self.fusion_blocks = [
            [
                layers.MultiHeadAttention(num_heads=num_heads, key_dim=hidden_dim),
                layers.LayerNormalization(),
                layers.Dense(hidden_dim * 4, activation='relu'),
                layers.Dense(hidden_dim),
                layers.Dropout(0.2)
            ] for _ in range(num_blocks)
        ]
        self.price_head = layers.Dense(1, activation='relu')
        self.variance_head = layers.Dense(1, activation='softplus')

    def call(self, inputs):
        text, image, tabular = inputs
        text_enc = self.text_projection(text) + self.text_embed
        image_enc = self.image_projection(image) + self.image_embed
        tabular_enc = self.tabular_projection(tabular) + self.tabular_embed
        combined = tf.stack([text_enc, image_enc, tabular_enc], axis=1)
        attn_output = self.attention(combined, combined)
        x = attn_output
        for attn, norm, dense1, dense2, dropout in self.fusion_blocks:
            x = norm(x + attn(x, x))
            x = norm(x + dropout(dense2(dense1(x))))
        x = tf.reduce_mean(x, axis=1)
        price = self.price_head(x)
        variance = self.variance_head(x)
        return price, variance

def smape_loss(y_true, y_pred):
    denominator = (tf.abs(y_true) + tf.abs(y_pred)) / 2.0
    diff = tf.abs(y_true - y_pred) / (denominator + 1e-10)
    return tf.reduce_mean(diff) * 100.0

m2tfm = M2TFM()
m2tfm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=[smape_loss, None])
with tf.device('/GPU:0' if len(physical_devices) > 0 else '/CPU:0'):
    m2tfm.fit(
        [train_text_embeddings, train_image_features, train_tabular],
        [train_df['price'].values, np.zeros(len(train_df))],
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        callbacks=[tf.keras.callbacks.ReduceLROnPlateau(patience=3)],
        verbose=1
    )
m2tfm_preds = m2tfm.predict([test_text_embeddings, test_image_features, test_tabular])[0]

# Step 5: Hybrid Ensemble
X = np.hstack([train_text_embeddings, train_image_features, train_tabular])
y = train_df['price'].values
lgb_preds = np.zeros(len(test_df))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    lgb_model = lgb.LGBMRegressor(n_estimators=1000, objective='mae', random_state=42)
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mae', early_stopping_rounds=50, verbose=False)
    lgb_preds += lgb_model.predict(np.hstack([test_text_embeddings, test_image_features, test_tabular])) / 5

xgb_model = xgb.XGBRegressor(n_estimators=1000, objective='reg:squarederror', random_state=42)
xgb_model.fit(X, y, eval_set=[(X, y)], eval_metric='mae', early_stopping_rounds=50, verbose=False)
xgb_preds = xgb_model.predict(np.hstack([test_text_embeddings, test_image_features, test_tabular]))

final_preds = 0.5 * m2tfm_preds.flatten() + 0.3 * lgb_preds + 0.2 * xgb_preds
final_preds = np.clip(final_preds, 0.01, 1000.0)

# Step 6: Output
output_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_preds
})
assert len(output_df) == len(test_df), "Output length mismatch"
assert output_df['price'].isna().sum() == 0, "NaN values detected"
assert (output_df['price'] > 0).all(), "Non-positive prices detected"
assert output_df['sample_id'].nunique() == len(test_df), "Duplicate sample_ids"
output_df.to_csv('submission.csv', index=False)

print("Prediction stats: min=", final_preds.min(), "max=", final_preds.max(), "mean=", final_preds.mean(), "std=", final_preds.std())